In [1]:
"""
Signaling game with occasionally observed states: equilibrium-aware analysis.

Classification uses *approximate strategic equilibrium* in the induced game,
not the payoff threshold and not a demand for pure sender strategies.

Status codes: 1 = near pooling equilibrium (stuck),
              0 = not near pooling equilibrium (unstuck),
             -1 = not applicable (SO at p=1: signaling is never used).

This is an approximate Nash test against one-player deviations, holding the
BP urn-total weights fixed; it is not a proof of stability of the urn-learning
ODE. The pooling and regret tolerances are reported and configurable.

Two output sweep sizes: n=4 and n=6. n=8 disabled by default.
"""

import numpy as np
import numba
from numba import njit, prange
import matplotlib.pyplot as plt
from dataclasses import dataclass
from enum import IntEnum
import time
from datetime import datetime
import os

# ---- Parallelism / reproducibility ----
NUM_THREADS = 4
BASE_SEED = 20261108  # -1 = random; any int >= 0 = reproducible results

# ---- Game parameters (used by RUN_SINGLE only) ----
N = 6
P_OBSERVE = 0
TIMESTEPS = 1_000_000
N_RUNS = 1000

# ---- Variant selection (used by RUN_SINGLE only) ----
ACTION_SELECTION = "SO"
REINFORCEMENT = "no_gating"

# ---- What to run ----
RUN_SINGLE = False
RUN_SCALING_TABLE = False
RUN_N4_SCALING = True
RUN_N6_SCALING = True
RUN_N8_SCALING = False
SCALING_T = 1_000_000
SCALING_RUNS = 1000
P_GRID = (0.0, 0.1, 0.3, 0.4, 0.5, 0.6, 0.7, 0.9, 1.0)   # 0.4/0.6 added around the p=0.5 threshold

# ---- Continuation of stuck runs ----
RUN_CONTINUATION = True
# Method 1 (mean-field): checkpoints as log10(total plays), started from T0
CONT_MF_LOG10_T = [7, 8, 9, 10, 12, 15, 20, 30, 50, 75, 100]
MF_EPS = 0.01                      # step = MF_EPS * t  (geometric time stepping)
# Method 2 (stochastic): checkpoints in total plays
CONT_STOCH_CHECKPOINTS = [2_000_000, 5_000_000, 10_000_000, 20_000_000, 50_000_000, 100_000_000]
CONT_STOCH_MAX_RUNS = 100          # at most this many stuck runs continued per cell
CONT_STOCH_VARIANTS = ("BP, gated",)   # None = all variants (expensive)

VARIANTS = [
    ("LP", "no_gating", 0, 0), ("LP", "gated", 0, 1),
    ("BP", "no_gating", 1, 0), ("BP", "gated", 1, 1),
    ("SO", "no_gating", 2, 0), ("SO", "gated", 2, 1),
]


class ActionSelectionEnum(IntEnum):
    LP = 0
    BP = 1
    SO = 2


class ReinforcementEnum(IntEnum):
    NO_GATING = 0
    GATED = 1


def parse_action_selection(s: str) -> ActionSelectionEnum:
    mapping = {"LP": ActionSelectionEnum.LP, "BP": ActionSelectionEnum.BP, "SO": ActionSelectionEnum.SO}
    return mapping[s.upper()]


def parse_reinforcement(s: str) -> ReinforcementEnum:
    mapping = {"NO_GATING": ReinforcementEnum.NO_GATING, "GATED": ReinforcementEnum.GATED}
    return mapping[s.upper().replace("-", "_")]


# Finite-sample decision tolerances. Explore their sensitivity when reporting.
# A signal counts as pooled only if it occurs appreciably and at least two
# states each have >=10% posterior probability conditional on that signal.
POOL_MIN_SIGNAL_MASS = 0.02
POOL_MIN_SECOND_POSTERIOR = 0.10
# Regret = potential conditional payoff gain from a unilateral best response.
# The sender, signal-receiver and state-receiver regrets must ALL be <= this.
EQUILIBRIUM_REGRET_TOL = 0.02

# Optional PAYOFF benchmark ONLY: never used to classify pooling / stuck.
SUCCESS_THRESHOLD = 0.875


def success_threshold(n: int) -> float:
    """Payoff benchmark only; NOT a pooling criterion."""
    return SUCCESS_THRESHOLD


def _status_rates(statuses):
    """Pooling / not pooling among evaluable runs; N/A returns NaN, not 100% unstuck."""
    valid = (statuses >= 0)
    if not np.any(valid):
        return np.nan, np.nan
    return float(np.mean(statuses[valid] == 1)), float(np.mean(statuses[valid] == 0))


def _recv_state_unused(p):
    return p <= 0.0


def _recv_signal_unused(a_sel, p):
    return a_sel == 2 and p >= 1.0


def _pct_or_na(v):
    return "n/a" if np.isnan(v) else f"{v*100:.1f}%"


def _mix_str(v, unused):
    return "unused" if (unused or np.isnan(v)) else f"{v:.2f}"


# ─────────────────────────────────────────────────────────────────────────────
# Core numba kernels
# ─────────────────────────────────────────────────────────────────────────────
@njit
def compute_normalized_entropy(probs):
    """Entropy normalised by log(N), ranges 0..1."""
    n = len(probs)
    if n <= 1: return 0.0
    ent = 0.0
    for p in probs:
        if p > 1e-12:
            ent -= p * np.log(p)
    return ent / np.log(n)


@njit
def avg_row_entropy(Q):
    n = Q.shape[0]
    s = 0.0
    for i in range(n):
        s += compute_normalized_entropy(Q[i] / Q[i].sum())
    return s / n


@njit
def compute_mixture_metrics(Q_sender, Q_recv_state, Q_recv_signal):
    """
    1) State Mixture:       avg entropy of P(signal|state)   [sender]
    2) Signal Mixture:      avg entropy of P(state|signal)   [sender inverted]
    3) Recv State Mixture:  avg entropy of P(action|state)   [receiver state urns]
    4) Recv Signal Mixture: avg entropy of P(action|signal)  [receiver signal urns]
    """
    n = Q_sender.shape[0]
    sender_strat = np.zeros((n, n))
    for i in range(n):
        sender_strat[i] = Q_sender[i] / Q_sender[i].sum()

    avg_state_mixture = avg_row_entropy(Q_sender)

    p_state_given_sig = np.zeros((n, n))   # [signal, state]
    for sig in range(n):
        col_sum = 0.0
        for st in range(n):
            col_sum += sender_strat[st, sig]
        if col_sum > 1e-12:
            for st in range(n):
                p_state_given_sig[sig, st] = sender_strat[st, sig] / col_sum
        else:
            for st in range(n):
                p_state_given_sig[sig, st] = 1.0 / n
    sum_sig_ent = 0.0
    for j in range(n):
        sum_sig_ent += compute_normalized_entropy(p_state_given_sig[j])
    avg_signal_mixture = sum_sig_ent / n

    return (avg_state_mixture, avg_signal_mixture,
            avg_row_entropy(Q_recv_state), avg_row_entropy(Q_recv_signal))


@njit
def choose_from_probs(probs):
    r = np.random.random()
    cumsum = 0.0
    for i in range(len(probs)):
        cumsum += probs[i]
        if r < cumsum: return i
    return len(probs) - 1


@njit
def play_steps(n, p_observe, action_selection, reinforcement, steps,
               Q_sender, Q_recv_signal, Q_recv_state):
    """
    Plays `steps` rounds IN PLACE on the three urn matrices; returns #successes.

    Learning rule
    -------------
    Unobserved play: receiver sees only the signal. On success reinforce
      sender (state,signal) and receiver-signal (signal,action).
    Observed play:
      SO : action = state-urn recommendation. The signal played no causal role,
           so sender and signal-urn are NOT reinforced; only the state urn is.
      LP/BP: action from the pooled urns; sender is reinforced on success.
           no_gating: both receiver urns reinforced on success.
           gated    : an urn is reinforced only if its own recommendation
                      (an independent draw from that urn) equals the action.
    """
    total_payoff = 0
    for t in range(steps):
        state = np.random.randint(0, n)
        sender_probs = Q_sender[state] / Q_sender[state].sum()
        signal = choose_from_probs(sender_probs)
        if np.random.random() < p_observe:
            probs_state = Q_recv_state[state] / Q_recv_state[state].sum()
            rec_state = choose_from_probs(probs_state)
            rec_signal = -1
            if action_selection == 2:
                action = rec_state
            else:
                probs_signal = Q_recv_signal[signal] / Q_recv_signal[signal].sum()
                rec_signal = choose_from_probs(probs_signal)
                if action_selection == 0:
                    pooled_probs = 0.5 * probs_state + 0.5 * probs_signal
                    action = choose_from_probs(pooled_probs)
                else:
                    pooled_Q = Q_recv_state[state] + Q_recv_signal[signal]
                    action = choose_from_probs(pooled_Q / pooled_Q.sum())
            if action == state:
                total_payoff += 1
                if action_selection == 2:
                    Q_recv_state[state, action] += 1      # state urn only
                else:
                    Q_sender[state, signal] += 1
                    if reinforcement == 0:
                        Q_recv_state[state, action] += 1
                        Q_recv_signal[signal, action] += 1
                    else:
                        if rec_state == action: Q_recv_state[state, action] += 1
                        if rec_signal == action: Q_recv_signal[signal, action] += 1
        else:
            probs_signal = Q_recv_signal[signal] / Q_recv_signal[signal].sum()
            action = choose_from_probs(probs_signal)
            if action == state:
                total_payoff += 1
                Q_sender[state, signal] += 1
                Q_recv_signal[signal, action] += 1
    return total_payoff


@njit
def run_single_simulation(n, p_observe, action_selection, reinforcement, timesteps):
    Q_sender = np.ones((n, n))
    Q_recv_signal = np.ones((n, n))
    Q_recv_state = np.ones((n, n))
    total_payoff = play_steps(n, p_observe, action_selection, reinforcement, timesteps,
                              Q_sender, Q_recv_signal, Q_recv_state)
    return Q_sender, Q_recv_signal, Q_recv_state, total_payoff


@njit
def compute_expected_payoff(n, p_observe, action_selection, Q_sender, Q_recv_signal, Q_recv_state):
    sender_strat = np.zeros((n, n))
    for i in range(n): sender_strat[i] = Q_sender[i] / Q_sender[i].sum()
    recv_signal_strat = np.zeros((n, n))
    recv_state_strat = np.zeros((n, n))
    for i in range(n):
        recv_signal_strat[i] = Q_recv_signal[i] / Q_recv_signal[i].sum()
        recv_state_strat[i] = Q_recv_state[i] / Q_recv_state[i].sum()
    exp_signal_only = 0.0
    for i in range(n):
        for j in range(n): exp_signal_only += sender_strat[i, j] * recv_signal_strat[j, i]
    exp_signal_only /= n
    exp_state_obs = 0.0
    for i in range(n):
        for j in range(n):
            p_sig = sender_strat[i, j]
            if action_selection == 2: p_correct = recv_state_strat[i, i]
            elif action_selection == 0: p_correct = 0.5 * recv_state_strat[i, i] + 0.5 * recv_signal_strat[j, i]
            else:
                Q_pooled = Q_recv_state[i] + Q_recv_signal[j]
                p_correct = Q_pooled[i] / Q_pooled.sum()
            exp_state_obs += p_sig * p_correct
    exp_state_obs /= n
    return (1 - p_observe) * exp_signal_only + p_observe * exp_state_obs


@njit
def pooling_equilibrium_diagnostics(n, p, asel, Qs, Qrs, Qrt):
    """Return (status, shared_signal, sender_regret, signal_regret, state_regret).

    status 1: approximate Nash equilibrium with a materially pooled signal;
           0: NOT an approximate pooling Nash equilibrium;
          -1: signaling never participates in play (SO, p=1).

    Freeze all urn row masses when testing unilateral strategy deviations.
    This matches BP's current count-weighted action mixing, but it is a
    *strategic* near-Nash condition, not a test for urn-ODE fixed points.

    Strategies: S[i,j] = P(signal=j | state=i)
                R[j,a] = P(action=a | signal=j)
                T[i,a] = P(action=a | observed state=i)

    Conditional action weights when state=i and signal=j:
          P(action) = alpha[i,j]*T[i] + beta[i,j]*R[j]
      SO: alpha=p, beta=1-p
      LP: alpha=p/2, beta=1-p/2
      BP: alpha=p*(sum(Qrt[i])/(sum(Qrt[i])+sum(Qrs[j]))),
          beta=1-alpha

    Receiver signal best response must maximize
        sum_i S[i,j] * beta[i,j] * R[j,i]
    (uniform state prior cancels). Receiver state best response is action i.
    Sender state i best response maximizes
        U[i,j] = alpha[i,j]*T[i,i] + beta[i,j]*R[j,i].
    A *mixed* sender is evaluated by its expected U, not by its argmax signal.
    """
    if asel == 2 and p >= 1.0:
        return -1, False, 0.0, 0.0, 0.0

    S = np.empty((n, n))
    R = np.empty((n, n))
    T = np.empty((n, n))
    state_masses = np.empty(n)
    sig_masses = np.empty(n)
    for i in range(n):
        sm = Qs[i].sum()
        rm = Qrs[i].sum()
        tm = Qrt[i].sum()
        state_masses[i] = tm
        sig_masses[i] = rm
        for j in range(n):
            S[i, j] = Qs[i, j] / sm
            R[i, j] = Qrs[i, j] / rm
            T[i, j] = Qrt[i, j] / tm

    alpha = np.empty((n, n))
    beta = np.empty((n, n))
    for i in range(n):
        for j in range(n):
            if asel == 2:
                a = p
            elif asel == 0:
                a = 0.5 * p
            else:
                a = p * state_masses[i] / (state_masses[i] + sig_masses[j])
            alpha[i, j] = a
            beta[i, j] = 1.0 - a

    # A signal is structurally pooled when its use is material AND more than
    # one state is a material source of that signal. This includes mixed pools.
    shared_signal = False
    for j in range(n):
        total_j = 0.0
        for i in range(n):
            total_j += S[i, j]
        marginal_j = total_j / n
        if marginal_j < POOL_MIN_SIGNAL_MASS:
            continue
        substantial_states = 0
        for i in range(n):
            if S[i, j] / total_j >= POOL_MIN_SECOND_POSTERIOR:
                substantial_states += 1
        if substantial_states >= 2:
            shared_signal = True

    # Sender mixed-strategy regret: compare its actual mixed payoff with its
    # most profitable pure-signal deviation, taking state observation into account.
    sender_regret = 0.0
    for i in range(n):
        chosen_utility = 0.0
        best_utility = -1.0
        for j in range(n):
            u = alpha[i, j] * T[i, i] + beta[i, j] * R[j, i]
            chosen_utility += S[i, j] * u
            if u > best_utility:
                best_utility = u
        regret = best_utility - chosen_utility
        if regret > sender_regret:
            sender_regret = regret

    # Signal receiver: all active message rows must put weight on optimal
    # actions under the *model-specific*, sometimes BP-weighted posterior.
    signal_regret = 0.0
    for j in range(n):
        signal_mass = 0.0
        weighted_sum = 0.0
        best_score = 0.0
        actual_score = 0.0
        for i in range(n):
            signal_mass += S[i, j] / n
            score = S[i, j] * beta[i, j]
            weighted_sum += score
            actual_score += score * R[j, i]
            if score > best_score:
                best_score = score
        if signal_mass >= POOL_MIN_SIGNAL_MASS and weighted_sum > 1e-14:
            regret = (best_score - actual_score) / weighted_sum
            if regret > signal_regret:
                signal_regret = regret

    # On an observed state i, the state-receiver policy is always best when
    # it chooses action i. At p=0 these rows have no effect: do not test them.
    state_regret = 0.0
    if p > 0.0:
        for i in range(n):
            regret = 1.0 - T[i, i]
            if regret > state_regret:
                state_regret = regret

    is_equilibrium = (sender_regret <= EQUILIBRIUM_REGRET_TOL and
                      signal_regret <= EQUILIBRIUM_REGRET_TOL and
                      state_regret <= EQUILIBRIUM_REGRET_TOL)
    status = 1 if (shared_signal and is_equilibrium) else 0
    return status, shared_signal, sender_regret, signal_regret, state_regret


@njit
def pooling_equilibrium_status(n, p, asel, Qs, Qrs, Qrt):
    """Return 1=stuck/near pooling NE, 0=unstuck, -1=not applicable."""
    return pooling_equilibrium_diagnostics(n, p, asel, Qs, Qrs, Qrt)[0]


@njit
def is_pooling_equilibrium(n, p, asel, Qs, Qrs, Qrt):
    """Boolean convenience helper; use status() to distinguish N/A from false."""
    return pooling_equilibrium_status(n, p, asel, Qs, Qrs, Qrt) == 1


def pooling_report(n, p, asel, Qs, Qrs, Qrt):
    """Human-readable diagnostics, useful for case studies / audit tests."""
    status, shared, sr, rr, tr = pooling_equilibrium_diagnostics(
        n, p, asel, Qs, Qrs, Qrt)
    return dict(status=int(status), shared_signal=bool(shared),
                sender_regret=float(sr), receiver_signal_regret=float(rr),
                receiver_state_regret=float(tr),
                equilibrium_regret_tolerance=EQUILIBRIUM_REGRET_TOL)


@njit
def all_metrics(n, p, asel, Qs, Qrs, Qrt):
    ep = compute_expected_payoff(n, p, asel, Qs, Qrs, Qrt)
    a, b, c, d = compute_mixture_metrics(Qs, Qrt, Qrs)
    return ep, a, b, c, d


@njit
def align_signals(sender_strat):
    """Greedy one-to-one signal relabeling, for display ONLY.

    WARNING: a pooled signal is shared by multiple states, so no one-to-one
    permutation can make every state's dominant signal equal its row index.
    NEVER use these averaged aligned matrices to classify pooling.
    """
    n = sender_strat.shape[0]
    perm = np.full(n, -1, dtype=np.int64)
    used_state = np.zeros(n, dtype=np.bool_)
    used_sig = np.zeros(n, dtype=np.bool_)
    for _ in range(n):
        best = -1.0
        bi = -1
        bj = -1
        for i in range(n):
            if used_state[i]:
                continue
            for j in range(n):
                if used_sig[j]:
                    continue
                if sender_strat[i, j] > best:
                    best = sender_strat[i, j]
                    bi = i
                    bj = j
        perm[bi] = bj
        used_state[bi] = True
        used_sig[bj] = True
    return perm


@njit(parallel=True)
def run_multiple_simulations_mats(n, p_observe, action_selection, reinforcement, timesteps,
                                  n_runs, seed=-1):
    """
    Returns per-run metrics, aligned strategy matrices and the final urn
    matrices (needed to continue stuck runs):
      (final_expected, avg_payoff, st_mix, sig_mix, rs_mix, rg_mix,
       sender_aligned, receiver_aligned, state_mat, Q_sender,
       Q_recv_signal, Q_recv_state, pooling_flags)
    """
    final_expected_payoffs = np.zeros(n_runs)
    average_payoffs        = np.zeros(n_runs)
    state_mixtures         = np.zeros(n_runs)
    signal_mixtures        = np.zeros(n_runs)
    recv_state_mixtures    = np.zeros(n_runs)
    recv_signal_mixtures   = np.zeros(n_runs)
    state_mat  = np.zeros((n_runs, n, n))
    sender_al  = np.zeros((n_runs, n, n))
    recv_al    = np.zeros((n_runs, n, n))
    Qs_all  = np.zeros((n_runs, n, n))
    Qrs_all = np.zeros((n_runs, n, n))
    Qrt_all = np.zeros((n_runs, n, n))
    pooling_flags = np.zeros(n_runs, dtype=np.int8)

    for run in prange(n_runs):
        if seed >= 0:
            np.random.seed(seed + run)
        Q_sender, Q_recv_signal, Q_recv_state, total_payoff = run_single_simulation(
            n, p_observe, action_selection, reinforcement, timesteps
        )
        final_expected_payoffs[run] = compute_expected_payoff(
            n, p_observe, action_selection, Q_sender, Q_recv_signal, Q_recv_state
        )
        average_payoffs[run] = total_payoff / timesteps
        a, b, c, d = compute_mixture_metrics(Q_sender, Q_recv_state, Q_recv_signal)
        state_mixtures[run] = a
        signal_mixtures[run] = b
        recv_state_mixtures[run] = c
        recv_signal_mixtures[run] = d
        Qs_all[run] = Q_sender
        Qrs_all[run] = Q_recv_signal
        Qrt_all[run] = Q_recv_state
        pooling_flags[run] = pooling_equilibrium_status(
            n, p_observe, action_selection, Q_sender, Q_recv_signal, Q_recv_state)

        S = np.zeros((n, n))
        R = np.zeros((n, n))
        T = np.zeros((n, n))
        for i in range(n):
            S[i] = Q_sender[i] / Q_sender[i].sum()
            R[i] = Q_recv_signal[i] / Q_recv_signal[i].sum()
            T[i] = Q_recv_state[i] / Q_recv_state[i].sum()
        perm = align_signals(S)
        for k in range(n):
            for j in range(n):
                state_mat[run, k, j] = T[k, j]
                sender_al[run, k, j] = S[k, perm[j]]   # column j = signal assigned to state j
                recv_al[run, k, j]   = R[perm[k], j]   # row k    = signal assigned to state k

    return (final_expected_payoffs, average_payoffs, state_mixtures, signal_mixtures,
            recv_state_mixtures, recv_signal_mixtures,
            sender_al, recv_al, state_mat, Qs_all, Qrs_all, Qrt_all, pooling_flags)


@njit(parallel=True)
def run_multiple_simulations(n, p_observe, action_selection, reinforcement, timesteps, n_runs,
                             seed=-1):
    final_expected_payoffs = np.zeros(n_runs)
    average_payoffs        = np.zeros(n_runs)
    state_mixtures         = np.zeros(n_runs)
    signal_mixtures        = np.zeros(n_runs)
    recv_state_mixtures    = np.zeros(n_runs)
    recv_signal_mixtures   = np.zeros(n_runs)
    pooling_flags          = np.zeros(n_runs, dtype=np.int8)

    for run in prange(n_runs):
        if seed >= 0:
            np.random.seed(seed + run)
        Q_sender, Q_recv_signal, Q_recv_state, total_payoff = run_single_simulation(
            n, p_observe, action_selection, reinforcement, timesteps
        )
        final_expected_payoffs[run] = compute_expected_payoff(
            n, p_observe, action_selection, Q_sender, Q_recv_signal, Q_recv_state
        )
        average_payoffs[run] = total_payoff / timesteps
        a, b, c, d = compute_mixture_metrics(Q_sender, Q_recv_state, Q_recv_signal)
        state_mixtures[run] = a
        signal_mixtures[run] = b
        recv_state_mixtures[run] = c
        recv_signal_mixtures[run] = d
        pooling_flags[run] = pooling_equilibrium_status(
            n, p_observe, action_selection, Q_sender, Q_recv_signal, Q_recv_state)

    return (final_expected_payoffs, average_payoffs, state_mixtures, signal_mixtures,
            recv_state_mixtures, recv_signal_mixtures, pooling_flags)


# ─────────────────────────────────────────────────────────────────────────────
# Continuation of stuck runs
# ─────────────────────────────────────────────────────────────────────────────
@njit
def expected_increments(n, p, asel, reinf, Qs, Qrs, Qrt):
    """
    Expected increment PER PLAY of every urn entry under the current strategies,
    matching play_steps exactly. Entries touched on a success (action == state = i):
        sender   dS[i, j]   (state i, signal j)
        sig urn  dRs[j, i]  (signal j, action i)
        st urn   dRt[i, i]
    w = P(state i, signal j) = S[i,j]/n ; pool = P(action == i | state i, signal j).
    """
    S = np.zeros((n, n))
    R = np.zeros((n, n))
    T = np.zeros((n, n))
    for i in range(n):
        S[i] = Qs[i] / Qs[i].sum()
        R[i] = Qrs[i] / Qrs[i].sum()
        T[i] = Qrt[i] / Qrt[i].sum()
    dS = np.zeros((n, n))
    dRs = np.zeros((n, n))
    dRt = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            w = S[i, j] / n
            # unobserved play: success prob = R[j, i]
            un = (1.0 - p) * w * R[j, i]
            dS[i, j] += un
            dRs[j, i] += un
            if asel == 2:
                dRt[i, i] += p * w * T[i, i]        # signal plays no role; action == rec_state
            else:
                if asel == 0:
                    pool = 0.5 * (T[i, i] + R[j, i])
                else:
                    pool = (Qrt[i, i] + Qrs[j, i]) / (Qrt[i].sum() + Qrs[j].sum())
                ob = p * w * pool
                dS[i, j] += ob
                if reinf == 0:
                    dRt[i, i] += ob
                    dRs[j, i] += ob
                else:                                # gated: own (independent) recommendation must match
                    dRt[i, i] += ob * T[i, i]
                    dRs[j, i] += ob * R[j, i]
    return dS, dRs, dRt


@njit
def meanfield_continue(n, p, asel, reinf, Qs0, Qrs0, Qrt0, t0, log10_targets, eps):
    """Method 1: advance urns by expected increments (geometric steps dt = eps*t)."""
    Qs = Qs0.copy()
    Qrs = Qrs0.copy()
    Qrt = Qrt0.copy()
    out = np.zeros((len(log10_targets), 6))
    t = t0
    for k in range(len(log10_targets)):
        target = 10.0 ** log10_targets[k]
        while t < target:
            dt = eps * t
            last = (t + dt >= target)
            if last:
                dt = target - t
            dS, dRs, dRt = expected_increments(n, p, asel, reinf, Qs, Qrs, Qrt)
            Qs += dt * dS
            Qrs += dt * dRs
            Qrt += dt * dRt
            if last:
                t = target
            else:
                t += dt
        ep, a, b, c, d = all_metrics(n, p, asel, Qs, Qrs, Qrt)
        out[k, 0] = ep
        out[k, 1] = a
        out[k, 2] = b
        out[k, 3] = c
        out[k, 4] = d
        out[k, 5] = pooling_equilibrium_status(n, p, asel, Qs, Qrs, Qrt)
    return out


@njit(parallel=True)
def meanfield_continue_many(n, p, asel, reinf, Qs_all, Qrs_all, Qrt_all, idx, t0,
                            log10_targets, eps):
    m = len(idx)
    out = np.zeros((m, len(log10_targets), 6))
    for k in prange(m):
        r = idx[k]
        out[k] = meanfield_continue(n, p, asel, reinf, Qs_all[r], Qrs_all[r], Qrt_all[r],
                                    t0, log10_targets, eps)
    return out


@njit
def stochastic_continue(n, p, asel, reinf, Qs0, Qrs0, Qrt0, t0, checkpoints):
    """Method 2: keep playing the stuck run; record metrics at each checkpoint."""
    Qs = Qs0.copy()
    Qrs = Qrs0.copy()
    Qrt = Qrt0.copy()
    out = np.zeros((len(checkpoints), 6))
    t = t0
    for k in range(len(checkpoints)):
        steps = checkpoints[k] - t
        if steps > 0:
            play_steps(n, p, asel, reinf, steps, Qs, Qrs, Qrt)
            t = checkpoints[k]
        ep, a, b, c, d = all_metrics(n, p, asel, Qs, Qrs, Qrt)
        out[k, 0] = ep
        out[k, 1] = a
        out[k, 2] = b
        out[k, 3] = c
        out[k, 4] = d
        out[k, 5] = pooling_equilibrium_status(n, p, asel, Qs, Qrs, Qrt)
    return out


@njit(parallel=True)
def stochastic_continue_many(n, p, asel, reinf, Qs_all, Qrs_all, Qrt_all, idx, t0,
                             checkpoints, seed):
    m = len(idx)
    out = np.zeros((m, len(checkpoints), 6))
    for k in prange(m):
        if seed >= 0:
            np.random.seed(seed + 1_000_003 + k)
        r = idx[k]
        out[k] = stochastic_continue(n, p, asel, reinf, Qs_all[r], Qrs_all[r], Qrt_all[r],
                                     t0, checkpoints)
    return out


def _summarize_continuation_metrics(raw, log10_times):
    """raw[m,checkpoint,0:5] metrics; raw[...,5] = status (-1/0/1)."""
    pooling = (raw[:, :, 5] == 1).mean(axis=0)
    return {
        "log10_t": log10_times,
        "n_runs": len(raw),
        "pooling": pooling,
        "unstuck": 1.0 - pooling,
        "metrics_mean": raw[:, :, :5].mean(axis=0),
    }


def run_continuation(n, p, a_sel, reinf, t0, Qs_all, Qrs_all, Qrt_all,
                     final_payoffs, pooling_flags, do_stoch):
    """Continue ONLY the runs that meet the pooling-equilibrium definition."""
    pooled_idx = np.flatnonzero(pooling_flags == 1).astype(np.int64)
    res = {"n_pooling": int(len(pooled_idx)), "pooling_idx": pooled_idx}
    if len(pooled_idx) == 0:
        return res
    res["payoff_t0"] = float(np.mean(final_payoffs[pooled_idx]))

    targets = np.array([x for x in CONT_MF_LOG10_T if 10.0 ** x > t0], dtype=np.float64)
    if len(targets):
        mf = meanfield_continue_many(n, p, a_sel, reinf, Qs_all, Qrs_all, Qrt_all, pooled_idx,
                                     float(t0), targets, MF_EPS)
        res["mf"] = _summarize_continuation_metrics(mf, targets)

    if do_stoch:
        sub = pooled_idx[:CONT_STOCH_MAX_RUNS]
        cps = np.array([c for c in CONT_STOCH_CHECKPOINTS if c > t0], dtype=np.int64)
        if len(cps):
            st = stochastic_continue_many(n, p, a_sel, reinf, Qs_all, Qrs_all, Qrt_all, sub,
                                          int(t0), cps, BASE_SEED)
            res["stoch"] = _summarize_continuation_metrics(
                st, np.log10(cps.astype(np.float64)))
    return res


def _write_cont_block(f, v_name, t, p, n_runs, cont, rs_u, rg_u):
    f.write(f"{'='*89}\nVARIANT: {v_name}   T0={_fmt_timestep(t)}   p={p:.1f}   "
            f"pooling/stuck at T0: {cont['n_pooling']}/{n_runs}\n{'='*89}\n")
    if cont["n_pooling"] == 0:
        f.write("  no pooling-equilibrium runs to continue\n\n")
        return
    f.write(f"  initial pooling cohort's mean expected payoff: {cont['payoff_t0']:.3f}\n")
    for key, title in (("mf", "METHOD 1  mean-field (expected-increment) continuation"),
                       ("stoch", "METHOD 2  stochastic continuation")):
        if key not in cont:
            continue
        c = cont[key]
        f.write(f"  {title}  [{c['n_runs']} initially pooling runs; "
                "Pool% = still pooling, Unst% = NOT pooling at this checkpoint]\n")
        f.write(f"  {'log10 t':>8} {'Pool%':>6} {'Unst%':>6} {'Pay':>6} "
                f"{'StMx':>6} {'SgMx':>6} {'RsMx':>6} {'RgMx':>6}\n")
        for k, lt in enumerate(c["log10_t"]):
            M = c["metrics_mean"][k]
            f.write(f"  {lt:8.2f} {c['pooling'][k]*100:5.1f}% {c['unstuck'][k]*100:5.1f}% "
                    f"{M[0]:6.3f} {M[1]:6.2f} {M[2]:6.2f} "
                    f"{_mix_str(M[3], rs_u):>6} {_mix_str(M[4], rg_u):>6}\n")
        f.write("\n")
    f.flush()


def _make_continuation_plot(cont_by_p, p_grid, key, v_name, n, t0, path):
    """Status plots are based on equilibrium classification, NOT payoff."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.7), facecolor="white")
    cmap = plt.cm.viridis
    plotted = False
    tick_labels = None
    for idx, p in enumerate(p_grid):
        c = cont_by_p.get(p)
        if not c or c["n_pooling"] == 0 or key not in c:
            continue
        exponents = [np.log10(t0)] + list(c[key]["log10_t"])
        # Equally spaced checkpoint positions preserve early-time visibility:
        # otherwise the T=1e6 to 1e8 changes disappear beside T=1e100.
        x = np.arange(len(exponents))
        if tick_labels is None:
            tick_labels = [f"1e{int(round(v))}" if abs(v - round(v)) < 0.01
                           else f"10^{v:.2f}" for v in exponents]
        still_pooled = [100.0] + list(c[key]["pooling"] * 100)
        unstuck = [0.0] + list(c[key]["unstuck"] * 100)
        payoff = [c["payoff_t0"]] + list(c[key]["metrics_mean"][:, 0])
        color = cmap(idx / max(1, len(p_grid) - 1))
        lab = f"p={p:.1f} (initial pooled={c['n_pooling']})"
        for ax, y in zip(axes, (still_pooled, unstuck, payoff)):
            ax.plot(x, y, marker="o", ms=3, color=color, label=lab)
        plotted = True
    if not plotted:
        for ax in axes:
            ax.set_axis_off()
        axes[1].text(0.5, 0.5, "No initially pooled runs for this variant and T0.\n"
                     "There is no pooling cohort to continue.",
                     ha="center", va="center", transform=axes[1].transAxes,
                     fontsize=11)
        fig.suptitle(f"{v_name} | n={n} | no initially pooled runs", fontsize=11)
        fig.savefig(path, dpi=130)
        plt.close(fig)
        return
    name = "mean-field" if key == "mf" else "stochastic"
    axes[0].set_ylabel("% STILL at a pooling equilibrium")
    axes[1].set_ylabel("% UNSTUCK (NOT pooling)")
    axes[2].set_ylabel("Mean expected payoff (not classification)")
    for ax in axes:
        ax.set_xlabel("Total plays (selected checkpoints; equally spaced)")
        ax.set_xticks(x)
        ax.set_xticklabels(tick_labels, rotation=40, ha="right", fontsize=8)
        ax.grid(True, alpha=0.3)
    axes[0].set_ylim(-3, 103)
    axes[1].set_ylim(-3, 103)
    axes[2].set_ylim(0, 1.02)
    axes[0].legend(fontsize=7, loc="best")
    fig.suptitle(
        f"{v_name} | n={n} | cohort initially at pooling equilibrium, T0={_fmt_timestep(t0)} ({name})",
        fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(path, dpi=130)
    plt.close(fig)


# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class GameConfig:
    n: int = 2
    p_observe: float = 0.1
    action_selection: ActionSelectionEnum = ActionSelectionEnum.LP
    reinforcement: ReinforcementEnum = ReinforcementEnum.NO_GATING
    timesteps: int = 10000
    n_runs: int = 100

    def variant_name(self) -> str:
        a_names = {0: "LP", 1: "BP", 2: "SO"}
        r_names = {0: "no_gating", 1: "gated"}
        return f"{a_names[int(self.action_selection)]}, {r_names[int(self.reinforcement)]}"

    @classmethod
    def from_globals(cls):
        return cls(n=N, p_observe=P_OBSERVE, timesteps=TIMESTEPS, n_runs=N_RUNS,
                   action_selection=parse_action_selection(ACTION_SELECTION),
                   reinforcement=parse_reinforcement(REINFORCEMENT))


def run_experiments(config: GameConfig, verbose: bool = True) -> dict:
    if verbose: print(f"Running {config.n_runs} simulations...", flush=True)
    start = time.time()
    final_payoffs, avg_payoffs, st_mixes, sig_mixes, rs_mixes, rg_mixes, pooling = run_multiple_simulations(
        config.n, config.p_observe, int(config.action_selection),
        int(config.reinforcement), config.timesteps, config.n_runs, BASE_SEED
    )
    if verbose: print(f"Completed in {time.time() - start:.2f}s", flush=True)
    return {
        'final_expected_payoffs':   final_payoffs,
        'average_payoffs':          avg_payoffs,
        'pooling_rate':             _status_rates(pooling)[0],
        'unstuck_rate':             _status_rates(pooling)[1],
        'payoff_success_rate':      np.mean(final_payoffs >= success_threshold(config.n)),
        'mean_final_expected':      np.mean(final_payoffs),
        'mean_average_payoff':      np.mean(avg_payoffs),
        'mean_state_mixture':       np.mean(st_mixes),
        'mean_signal_mixture':      np.mean(sig_mixes),
        'mean_recv_state_mixture':  np.mean(rs_mixes),
        'mean_recv_signal_mixture': np.mean(rg_mixes),
    }


def _fmt_timestep(t: int) -> str:
    exponent = round(np.log10(t))
    if 10 ** exponent == t: return f"1e{exponent}"
    return f"{t:,}"


def _save_cell_distribution_plot(final_payoffs, variant_name, n, timesteps, p_obs, n_runs,
                                 unstuck_rate, mean_avg_payoff, st_mix, sig_mix, rs_str, rg_str,
                                 save_path):
    plt.style.use("default")
    fig, ax = plt.subplots(figsize=(8, 5), facecolor="white")
    ax.hist(final_payoffs, bins=20, density=True, alpha=0.7, color="steelblue", edgecolor="white")
    thr = success_threshold(n)
    ax.axvline(x=thr, color="red", linestyle="--", label=f"Payoff reference {thr:.3f} (NOT pooling rule)")
    ax.set_title(
        f"{variant_name} | n={n}, T={timesteps:,}, p={p_obs:.1f}\n"
        f"Unstuck (NOT pooling): {'n/a' if np.isnan(unstuck_rate) else f'{unstuck_rate*100:.1f}%'} | Avg Payoff: {mean_avg_payoff:.3f}\n"
        f"State Mix: {st_mix:.3f} | Signal Mix: {sig_mix:.3f} | "
        f"Recv-State Mix: {rs_str} | Recv-Signal Mix: {rg_str}",
        fontsize=9
    )
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1.05)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.close()


def _n_runs_for_timestep(t: int) -> int:
    return 100 if t >= 10_000_000 else 1_000


# ─────────────────────────────────────────────────────────────────────────────
# Table formatting shared by both table writers
# ─────────────────────────────────────────────────────────────────────────────
CELL_W = 34
SUB_HEADER = f"{'Unst%':>5} {'Pay':>4} {'StMx':>4} {'SgMx':>4} {'RsMx':>6} {'RgMx':>6}"


def _cell_text(unstuck_fraction, pay, st, sg, rs, rg, rs_u, rg_u):
    rate_text = "n/a" if np.isnan(unstuck_fraction) else f"{unstuck_fraction*100:3.0f}%"
    return (f"{rate_text:>4} {pay:4.2f} {st:4.2f} {sg:4.2f} "
            f"{_mix_str(rs, rs_u):>6} {_mix_str(rg, rg_u):>6}")


def run_scaling_table(n=4, save_dir="scaling_results"):
    t_grid = [100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000]
    p_grid = [0.0, 0.2, 0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 1.0]
    os.makedirs(save_dir, exist_ok=True)
    summary_path = os.path.join(save_dir, "all_metrics_summary.txt")
    PLAYS_W = 8
    total_w = PLAYS_W + 3 + len(p_grid) * (CELL_W + 3)
    with open(summary_path, "w") as f_sum:
        f_sum.write(f"SCALING TABLE  n={n}  {datetime.now()}\n")
        f_sum.write("Metrics: Unst%=NOT at pooling equilibrium  Pay=avg payoff  StMx=state mixture  SgMx=signal mixture\n"
                    "         RsMx=recv-state mixture  RgMx=recv-signal mixture  (0=deterministic, 1=uniform)\n"
                    "         'unused' = receiver urn not updated; n/a = SO p=1 signaling channel inactive\n\n")
        for action_label, reinf_label, a_sel, reinf in VARIANTS:
            v_name = f"{action_label}, {reinf_label}"
            f_sum.write(f"{'='*total_w}\nVARIANT: {v_name}\n{'='*total_w}\n")
            f_sum.write(f"{'Plays':<{PLAYS_W}} | " + " | ".join(f"p={p:.1f}".center(CELL_W) for p in p_grid) + " |\n")
            f_sum.write(f"{'':>{PLAYS_W}} | " + " | ".join(SUB_HEADER.center(CELL_W) for _ in p_grid) + " |\n")
            f_sum.write("-" * total_w + "\n")
            for t in t_grid:
                runs = _n_runs_for_timestep(t)
                row = f"{_fmt_timestep(t):<{PLAYS_W}} | "
                for p in p_grid:
                    res, avgs, st, sig, rs, rg, pooling = run_multiple_simulations(
                        n, p, a_sel, reinf, t, runs, BASE_SEED)
                    sr = _status_rates(pooling)[1]
                    rs_u, rg_u = _recv_state_unused(p), _recv_signal_unused(a_sel, p)
                    row += f"{_cell_text(sr, np.mean(avgs), np.mean(st), np.mean(sig), np.mean(rs), np.mean(rg), rs_u, rg_u):<{CELL_W}} | "
                    dist_dir = os.path.join(save_dir, "plots", v_name.replace(", ", "_"))
                    os.makedirs(dist_dir, exist_ok=True)
                    _save_cell_distribution_plot(
                        res, v_name, n, t, p, runs, sr, np.mean(avgs), np.mean(st), np.mean(sig),
                        _mix_str(np.mean(rs), rs_u), _mix_str(np.mean(rg), rg_u),
                        os.path.join(dist_dir, f"T{_fmt_timestep(t)}_p{p:.1f}.png"))
                f_sum.write(row + "\n")
            f_sum.write("\n")
    print(f"Scaling table saved to: {summary_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Matrix output (aligned sender / state / receiver) for every (variant, T, p)
# ─────────────────────────────────────────────────────────────────────────────
def _matrix_to_text(M, row_label, col_label):
    n = M.shape[0]
    head = f"{row_label} \\ {col_label}"
    lines = [f"    {head}", "        " + " ".join(f"{j:>5d}" for j in range(n))]
    for i in range(n):
        lines.append(f"    {i:>3d} " + " ".join(f"{M[i, j]:5.2f}" for j in range(n)))
    return "\n".join(lines)


def _draw_matrix(ax, M, title, row_label, col_label):
    n = M.shape[0]
    ax.imshow(M, vmin=0, vmax=1, cmap="Blues")
    ax.set_title(title, fontsize=9)
    ax.set_xlabel(col_label, fontsize=8)
    ax.set_ylabel(row_label, fontsize=8)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.tick_params(labelsize=7)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if M[i, j] > 0.55 else "black")


def _save_matrix_figure(mats, v_name, n, t, p, n_runs, path, state_unused=False, recv_unused=False):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), facecolor="white")
    _draw_matrix(axes[0], mats["sender"], "1. Sender (aligned)\nP(sent | seen)",
                 "state seen", "signal sent (relabelled)")
    _draw_matrix(axes[1], mats["state"],
                 "2. State\nP(act | obs)" + ("   [unused: never updated]" if state_unused else ""),
                 "state observed", "action")
    _draw_matrix(axes[2], mats["receiver"],
                 "3. Receiver (aligned)\nP(act | rec)" + ("   [unused: never updated]" if recv_unused else ""),
                 "signal received (relabelled)", "action")
    fig.suptitle(f"{v_name} | n={n}, T={_fmt_timestep(t)}, p={p:.1f} | mean of {n_runs} runs",
                 fontsize=11)
    plt.tight_layout(rect=(0, 0, 1, 0.94))
    plt.savefig(path, dpi=110)
    plt.close()


def _make_sweep_plot(all_results, p_grid, t, n, metric_keys, ylabel,
                     title_suffix, filepath, label_map=None):
    """One line per variant. If two metric keys are given, the second is fainter.
    NaN values ('unused') leave a gap."""
    COLOR_MAP  = {"LP": "tab:blue", "BP": "tab:orange", "SO": "tab:green"}
    DASH_MAP   = {"no_gating": "-", "gated": "--"}
    MARKER_MAP = {"no_gating": "o", "gated": "s"}
    if isinstance(metric_keys, str):
        metric_keys = [metric_keys]

    fig, ax = plt.subplots(figsize=(9, 5), facecolor="white")
    for action_label, reinf_label, _, _ in VARIANTS:
        v_name = f"{action_label}, {reinf_label}"
        for idx, mkey in enumerate(metric_keys):
            vals = [all_results[v_name][t][p][mkey] for p in p_grid]
            name = f"{v_name}" if idx == 0 else f"  ↳ {label_map[mkey]} ({v_name})"
            ax.plot(p_grid, vals,
                    color=COLOR_MAP[action_label], linestyle=DASH_MAP[reinf_label],
                    marker=MARKER_MAP[reinf_label], markersize=5,
                    linewidth=1.8 if idx == 0 else 1.2,
                    alpha=1.0 if idx == 0 else 0.45, label=name)
    ax.set_xlabel("p_observe", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f"n={n}, T={_fmt_timestep(t)} — {title_suffix}\n"
                 "Color: LP=blue  BP=orange  SO=green   Line: solid=no_gating  dashed=gated",
                 fontsize=9)
    ax.legend(fontsize=7, ncol=2, loc="best")
    ax.set_xlim(-0.02, 1.02)
    if len(metric_keys) == 1 and metric_keys[0] in ("pooling_rate", "unstuck_rate"):
        # Do not render SO p=1 as 100% unstuck; it is not observable in play.
        ax.set_ylim(-0.02, 1.02)
        ax.set_yticks(np.linspace(0, 1, 6))
        ax.text(0.01, -0.14, "SO at p=1: N/A (no signaling channel in play)",
                transform=ax.transAxes, fontsize=8, va="top")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(filepath, dpi=130)
    plt.close()


def run_scaling_sweep(n: int, t_grid: list, n_runs: int = 1000,
                      save_dir: str = "scaling_sweep",
                      p_grid=None,
                      make_plots: bool = True,
                      save_matrices: bool = True,
                      do_continuation=None) -> dict:
    """
    Runs all 6 variants for every (T, p), writes a summary table, sweep graphs,
    the three aligned mean strategy matrices, and (optionally) the continuation
    of pooling-equilibrium (stuck) runs by mean-field and stochastic methods.
    Returns all_results[variant][T][p] -> dict of metrics.
    """
    if p_grid is None:
        p_grid = P_GRID
    if do_continuation is None:
        do_continuation = RUN_CONTINUATION
    p_grid = list(p_grid)
    thr = success_threshold(n)

    os.makedirs(save_dir, exist_ok=True)
    summary_path = os.path.join(save_dir, f"n{n}_summary.txt")
    matrices_txt_path = os.path.join(save_dir, f"n{n}_matrices.txt")
    matrices_npz_path = os.path.join(save_dir, f"n{n}_matrices.npz")
    cont_path = os.path.join(save_dir, f"n{n}_continuation.txt")
    matrix_plot_dir = os.path.join(save_dir, "matrices")
    if save_matrices:
        os.makedirs(matrix_plot_dir, exist_ok=True)

    PLAYS_W = 8
    total_w = PLAYS_W + 3 + len(p_grid) * (CELL_W + 3)
    div = "-" * total_w
    bar = "=" * total_w

    all_results = {}
    npz_store = {}

    f_mat = open(matrices_txt_path, "w") if save_matrices else None
    if f_mat:
        f_mat.write(f"MEAN STRATEGY MATRICES  n={n}  {datetime.now()}\n"
                    f"Each matrix is averaged over {n_runs} runs. Rows sum to 1.\n"
                    "Sender   : rows = state seen,      cols = signal sent   (aligned)\n"
                    "State    : rows = state observed,  cols = action\n"
                    "Receiver : rows = signal received, cols = action        (aligned)\n"
                    "'aligned' = greedy one-to-one signal relabeling (a display aid only).\n"
                    "Pooling cannot be recovered reliably from averages of these aligned matrices.\n\n")
    f_cont = open(cont_path, "w") if do_continuation else None
    if f_cont:
        f_cont.write(f"CONTINUATION OF INITIALLY POOLING (STUCK) RUNS  n={n}  {datetime.now()}\n"
                     "Stuck = near pooling equilibrium; unstuck = NOT near pooling equilibrium; SO p=1=N/A.\n"
                     f"Test: signal mass >= {POOL_MIN_SIGNAL_MASS:.2f}, second posterior >= {POOL_MIN_SECOND_POSTERIOR:.2f}, "
                     f"sender AND both receiver regrets <= {EQUILIBRIUM_REGRET_TOL:.2f}.\n"
                     f"Method 1: urns advanced by expected increments (eps={MF_EPS}).\n"
                     f"Method 2: stuck runs continued stochastically (max {CONT_STOCH_MAX_RUNS} runs/cell, "
                     f"variants: {CONT_STOCH_VARIANTS}).\n\n")

    with open(summary_path, "w") as f:
        f.write(f"SCALING TABLE  n={n}  {datetime.now()}\n")
        f.write("Metrics: Unst%=NOT at pooling equilibrium  Pay=avg payoff  StMx=state mixture  SgMx=signal mixture\n"
                "         RsMx=recv-state mixture  RgMx=recv-signal mixture  (0=deterministic, 1=uniform)\n"
                "         'unused' = receiver urn not updated; n/a = SO p=1 signaling channel inactive\n")
        f.write("Stuck = near pooling equilibrium, unstuck = not near pooling equilibrium; SO p=1 is N/A.\n")
        f.write(f"Pooling test: signal mass >= {POOL_MIN_SIGNAL_MASS:.2f}, second posterior >= {POOL_MIN_SECOND_POSTERIOR:.2f}, "
                f"all regrets <= {EQUILIBRIUM_REGRET_TOL:.2f}.\n")
        f.write(f"Payoff reference only: {thr:.4f}. Runs per cell: {n_runs}\n\n")

        for action_label, reinf_label, a_sel, reinf in VARIANTS:
            v_name = f"{action_label}, {reinf_label}"
            v_tag = v_name.replace(", ", "_")
            all_results[v_name] = {}

            header1 = f"{'Plays':<{PLAYS_W}} | " + " | ".join(f"p={p:.1f}".center(CELL_W) for p in p_grid) + " |"
            header2 = f"{'':>{PLAYS_W}} | " + " | ".join(SUB_HEADER.center(CELL_W) for _ in p_grid) + " |"
            block = [bar, f"VARIANT: {v_name}", bar, header1, header2, div]
            print(f"\nVARIANT: {v_name}", flush=True)

            for t in t_grid:
                all_results[v_name][t] = {}
                row = f"{_fmt_timestep(t):<{PLAYS_W}} | "

                for p in p_grid:
                    (res, avgs, st, sig, rs, rg,
                     S_al, R_al, St, Qs_all, Qrs_all, Qrt_all, pooling) = run_multiple_simulations_mats(
                        n, p, a_sel, reinf, t, n_runs, BASE_SEED
                    )
                    rs_u, rg_u = _recv_state_unused(p), _recv_signal_unused(a_sel, p)
                    mats = {"sender": S_al.mean(axis=0),
                            "state": St.mean(axis=0),
                            "receiver": R_al.mean(axis=0)}
                    m = {
                        "pooling_rate":             _status_rates(pooling)[0],
                        "unstuck_rate":             _status_rates(pooling)[1],
                        "payoff_success_rate":      float(np.mean(res >= thr)),
                        "mean_avg_payoff":          float(np.mean(avgs)),
                        "mean_exp_payoff":          float(np.mean(res)),
                        "mean_state_mixture":       float(np.mean(st)),
                        "mean_signal_mixture":      float(np.mean(sig)),
                        "mean_recv_state_mixture":  np.nan if rs_u else float(np.mean(rs)),
                        "mean_recv_signal_mixture": np.nan if rg_u else float(np.mean(rg)),
                        "final_payoffs":            res,
                        "pooling_flags":            pooling,
                        "matrices":                 mats,
                    }
                    all_results[v_name][t][p] = m

                    cell = _cell_text(m["unstuck_rate"], m["mean_avg_payoff"], m["mean_state_mixture"],
                                      m["mean_signal_mixture"], m["mean_recv_state_mixture"],
                                      m["mean_recv_signal_mixture"], rs_u, rg_u)
                    row += f"{cell:<{CELL_W}} | "

                    if save_matrices:
                        key = f"{v_tag}__T{_fmt_timestep(t)}__p{p:.1f}"
                        for name, M in mats.items():
                            npz_store[f"{key}__{name}"] = M
                        f_mat.write(f"{'='*60}\nVARIANT: {v_name}   T={_fmt_timestep(t)}   p={p:.1f}\n{'='*60}\n")
                        f_mat.write("1. SENDER (aligned)  P(sent | seen)\n" +
                                    _matrix_to_text(mats["sender"], "seen", "sent") + "\n")
                        f_mat.write("2. STATE  P(act | obs)" + ("   [unused]" if rs_u else "") + "\n" +
                                    _matrix_to_text(mats["state"], "obs", "act") + "\n")
                        f_mat.write("3. RECEIVER (aligned)  P(act | rec)" + ("   [unused]" if rg_u else "") + "\n" +
                                    _matrix_to_text(mats["receiver"], "rec", "act") + "\n\n")
                        f_mat.flush()
                        vdir = os.path.join(matrix_plot_dir, v_tag)
                        os.makedirs(vdir, exist_ok=True)
                        _save_matrix_figure(mats, v_name, n, t, p, n_runs,
                                            os.path.join(vdir, f"T{_fmt_timestep(t)}_p{p:.1f}.png"),
                                            state_unused=rs_u, recv_unused=rg_u)

                    if do_continuation:
                        do_stoch = (CONT_STOCH_VARIANTS is None) or (v_name in CONT_STOCH_VARIANTS)
                        cont = run_continuation(n, p, a_sel, reinf, t, Qs_all, Qrs_all, Qrt_all,
                                                res, pooling, do_stoch)
                        m["continuation"] = cont
                        _write_cont_block(f_cont, v_name, t, p, n_runs, cont, rs_u, rg_u)

                block.append(row)
                print(f"  T={_fmt_timestep(t)} done", flush=True)

            block.append("")
            text = "\n".join(block) + "\n"
            f.write(text)
            f.flush()
            print(text)

    if f_mat:
        f_mat.close()
        np.savez_compressed(matrices_npz_path, **npz_store)
        print(f"Matrices saved to: {matrices_txt_path}, {matrices_npz_path}, {matrix_plot_dir}/")
    if f_cont:
        f_cont.close()
        print(f"Continuation results saved to: {cont_path}")

    print(f"Summary saved to: {summary_path}")

    if make_plots:
        plots_dir = os.path.join(save_dir, "plots")
        os.makedirs(plots_dir, exist_ok=True)
        for t in t_grid:
            tl = _fmt_timestep(t)
            _make_sweep_plot(all_results, p_grid, t, n, "unstuck_rate",
                             "Unstuck fraction (NOT pooling)", "Unstuck vs p_observe",
                             os.path.join(plots_dir, f"T{tl}_unstuck_rate.png"))
            _make_sweep_plot(all_results, p_grid, t, n, "pooling_rate",
                             "Stuck fraction (at pooling equilibrium)", "Pooling vs p_observe",
                             os.path.join(plots_dir, f"T{tl}_pooling_rate.png"))
            _make_sweep_plot(all_results, p_grid, t, n, "mean_avg_payoff",
                             "Avg Payoff (over run)", "Average Payoff vs p_observe",
                             os.path.join(plots_dir, f"T{tl}_avg_payoff.png"))
            _make_sweep_plot(all_results, p_grid, t, n, "mean_exp_payoff",
                             "Expected Payoff (final strategy)", "Expected Payoff vs p_observe",
                             os.path.join(plots_dir, f"T{tl}_exp_payoff.png"))
            _make_sweep_plot(all_results, p_grid, t, n,
                             ["mean_state_mixture", "mean_signal_mixture"],
                             "Normalized Entropy", "Sender Entropy vs p_observe",
                             os.path.join(plots_dir, f"T{tl}_sender_entropy.png"),
                             label_map={"mean_state_mixture": "StateMix",
                                        "mean_signal_mixture": "SigMix"})
            _make_sweep_plot(all_results, p_grid, t, n, "mean_recv_state_mixture",
                             "Normalized Entropy", "Receiver State-Urn Entropy vs p_observe (p=0 unused)",
                             os.path.join(plots_dir, f"T{tl}_receiver_entropy.png"))
            _make_sweep_plot(all_results, p_grid, t, n, "mean_recv_signal_mixture",
                             "Normalized Entropy", "Receiver Signal-Urn Entropy vs p_observe (SO p=1 unused)",
                             os.path.join(plots_dir, f"T{tl}_receiver_signal_entropy.png"))
        print(f"Plots saved to: {plots_dir}")

        if do_continuation:
            cdir = os.path.join(save_dir, "continuation_plots")
            os.makedirs(cdir, exist_ok=True)
            for action_label, reinf_label, _, _ in VARIANTS:
                v_name = f"{action_label}, {reinf_label}"
                v_tag = v_name.replace(", ", "_")
                for t in t_grid:
                    by_p = {p: all_results[v_name][t][p]["continuation"] for p in p_grid}
                    for key, nm in (("mf", "meanfield"), ("stoch", "stochastic")):
                        _make_continuation_plot(by_p, p_grid, key, v_name, n, t,
                                                os.path.join(cdir, f"{v_tag}_T{_fmt_timestep(t)}_{nm}.png"))
            print(f"Continuation plots saved to: {cdir}")

    return all_results


def run_n4_scaling(n_runs: int = SCALING_RUNS, save_dir: str = "n4_scaling") -> dict:
    return run_scaling_sweep(4, [SCALING_T], n_runs, save_dir)


def run_n6_scaling(n_runs: int = SCALING_RUNS, save_dir: str = "n6_scaling") -> dict:
    return run_scaling_sweep(6, [SCALING_T], n_runs, save_dir)


def run_n8_scaling(n_runs: int = SCALING_RUNS, save_dir: str = "n8_scaling") -> dict:
    return run_scaling_sweep(8, [SCALING_T], n_runs, save_dir)


def main():
    if NUM_THREADS is not None:
        numba.set_num_threads(min(NUM_THREADS, numba.config.NUMBA_NUM_THREADS))
    print(f"Numba threads in use: {numba.get_num_threads()} "
          f"(max {numba.config.NUMBA_NUM_THREADS}), threading layer requested: "
          f"{numba.config.THREADING_LAYER}")
    session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_dir = f"session_{session_id}"
    os.makedirs(base_dir, exist_ok=True)
    print(f"All results will be saved in: {base_dir}")

    # JIT warmup (all kernels)
    print("Warming up JIT...", end=" ", flush=True)
    run_multiple_simulations(2, 0.1, 0, 0, 100, 2, BASE_SEED)
    out = run_multiple_simulations_mats(2, 0.1, 0, 0, 100, 2, BASE_SEED)
    idx = np.array([0, 1], dtype=np.int64)
    meanfield_continue_many(2, 0.1, 0, 0, out[9], out[10], out[11], idx, 100.0,
                            np.array([3.0, 4.0]), MF_EPS)
    stochastic_continue_many(2, 0.1, 0, 0, out[9], out[10], out[11], idx, 100,
                             np.array([200, 300], dtype=np.int64), BASE_SEED)
    print("Done.\n")

    if RUN_SINGLE:
        config = GameConfig.from_globals()
        results = run_experiments(config)
        p = config.p_observe
        print(f"\nResults for {config.variant_name()}:")
        print(f"  Stuck (pooling):      {_pct_or_na(results['pooling_rate'])}")
        print(f"  Unstuck (not pooling): {_pct_or_na(results['unstuck_rate'])}")
        print(f"  Payoff >= {SUCCESS_THRESHOLD}:  {results['payoff_success_rate']*100:.1f}%")
        print(f"  Avg Payoff:           {results['mean_average_payoff']:.4f}")
        print(f"  State Mixture:        {results['mean_state_mixture']:.4f}")
        print(f"  Signal Mixture:       {results['mean_signal_mixture']:.4f}")
        print(f"  Recv-State Mixture:   {_mix_str(results['mean_recv_state_mixture'], _recv_state_unused(p))}")
        print(f"  Recv-Signal Mixture:  {_mix_str(results['mean_recv_signal_mixture'], _recv_signal_unused(int(config.action_selection), p))}")

    if RUN_SCALING_TABLE:
        print("\nRunning Scaling Tables (this may take a while)...")
        run_scaling_table(n=N, save_dir=os.path.join(base_dir, "scaling"))

    if RUN_N4_SCALING:
        run_n4_scaling(save_dir=os.path.join(base_dir, "n4_scaling"))
    if RUN_N6_SCALING:
        run_n6_scaling(save_dir=os.path.join(base_dir, "n6_scaling"))
    if RUN_N8_SCALING:
        run_n8_scaling(save_dir=os.path.join(base_dir, "n8_scaling"))


if __name__ == "__main__":
    main()

Numba threads in use: 4 (max 10), threading layer requested: default
All results will be saved in: session_20261115_214150
Warming up JIT... Done.


VARIANT: LP, no_gating
  T=1e6 done
VARIANT: LP, no_gating
Plays    |               p=0.0                |               p=0.1                |               p=0.3                |               p=0.4                |               p=0.5                |               p=0.6                |               p=0.7                |               p=0.9                |               p=1.0                |
         | Unst%  Pay StMx SgMx   RsMx   RgMx | Unst%  Pay StMx SgMx   RsMx   RgMx | Unst%  Pay StMx SgMx   RsMx   RgMx | Unst%  Pay StMx SgMx   RsMx   RgMx | Unst%  Pay StMx SgMx   RsMx   RgMx | Unst%  Pay StMx SgMx   RsMx   RgMx | Unst%  Pay StMx SgMx   RsMx   RgMx | Unst%  Pay StMx SgMx   RsMx   RgMx | Unst%  Pay StMx SgMx   RsMx   RgMx |
--------------------------------------------------------------------------------------------------------